# Семинар 5-СС. Өз бетінше шешуге арналған есептер

Барлық қабаттар **numpy-мен нөлден жазылды** (дайын `conv`/`pool`/`batchnorm` қолданылмайды).
Жауап алдымен формуламен есептеледі, содан кейін өз қабатымызды нақты іске қосып тексереміз.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)


def conv2d(x, W, b, stride=1, pad=0, dil=1):
    """x: (H, W, Cin), W: (k, k, Cin, F), b: (F,) -> (Ho, Wo, F)"""
    k, _, cin, f = W.shape
    k_eff = dil * (k - 1) + 1
    xp = np.pad(x, ((pad, pad), (pad, pad), (0, 0)))
    ho = (xp.shape[0] - k_eff) // stride + 1
    wo = (xp.shape[1] - k_eff) // stride + 1
    out = np.zeros((ho, wo, f))
    for i in range(ho):
        for j in range(wo):
            patch = xp[i*stride : i*stride + k_eff : dil,
                       j*stride : j*stride + k_eff : dil, :]      # (k, k, Cin)
            out[i, j] = np.tensordot(patch, W, axes=([0, 1, 2], [0, 1, 2])) + b
    return out


def maxpool2d(x, k, stride, pad=0):
    xp = np.pad(x, ((pad, pad), (pad, pad), (0, 0)), constant_values=-np.inf)
    ho = (xp.shape[0] - k) // stride + 1
    wo = (xp.shape[1] - k) // stride + 1
    out = np.zeros((ho, wo, x.shape[2]))
    for i in range(ho):
        for j in range(wo):
            out[i, j] = xp[i*stride : i*stride + k, j*stride : j*stride + k].max(axis=(0, 1))
    return out


class BatchNorm:
    def __init__(self, c, eps=1e-5, momentum=0.1):
        self.gamma = np.ones(c)            # оқытылатын
        self.beta = np.zeros(c)            # оқытылатын
        self.running_mean = np.zeros(c)    # статистика
        self.running_var = np.ones(c)      # статистика
        self.eps, self.momentum = eps, momentum

    def __call__(self, x):                 # x: (H, W, C), оқыту режимі
        mean = x.mean(axis=(0, 1))
        var = x.var(axis=(0, 1))
        m = self.momentum
        self.running_mean = (1 - m) * self.running_mean + m * mean
        self.running_var = (1 - m) * self.running_var + m * var
        return self.gamma * (x - mean) / np.sqrt(var + self.eps) + self.beta

    def n_trainable(self):
        return self.gamma.size + self.beta.size

    def n_stats(self):
        return self.running_mean.size + self.running_var.size

## Есеп 1

Кіріс $128\times128\times3$, Conv $7\times7$, 64 сүзгі, $S=2$, $P=3$.

$$H_{out}=\left\lfloor\frac{H+2P-k}{S}\right\rfloor+1,\qquad \text{params}=(k\cdot k\cdot C_{in}+1)\cdot F$$

In [ ]:
H, Cin, K, F, S, P = 128, 3, 7, 64, 2, 3

# формула бойынша
h_formula = (H + 2 * P - K) // S + 1
p_formula = (K * K * Cin + 1) * F
print(f"Формула: {h_formula}x{h_formula}x{F}, параметр = ({K*K*Cin}+1)*{F} = {p_formula}")

# өз conv2d қабатымызбен
x = rng.standard_normal((H, H, Cin))
W1 = rng.standard_normal((K, K, Cin, F)) * 0.05
b1 = np.zeros(F)
y1 = conv2d(x, W1, b1, stride=S, pad=P)
print(f"conv2d нәтижесі: {y1.shape}, параметр = {W1.size + b1.size}")

## Есеп 2

Одан кейін MaxPool $3\times3$, $S=2$, $P=1$. Шығыс өлшемі және екі қабаттан кейінгі RF.

RF рекуррентті: $RF \mathrel{+}= (k_{eff}-1)\cdot jump$, $jump \mathrel{*}= S$.

In [ ]:
y2 = maxpool2d(y1, k=3, stride=2, pad=1)
print(f"MaxPool шығысы: {y2.shape}")

# RF формуламен
rf, jump = 1, 1
for name, k, s in [("Conv 7x7 S=2", 7, 2), ("MaxPool 3x3 S=2", 3, 2)]:
    rf += (k - 1) * jump
    jump *= s
    print(f"{name} кейін RF = {rf}")


def rf_empirical(layers, n_in, dil_list=None):
    """Бір арналы, оң салмақты (бірлік) қабаттар тізбегі. Ортаңғы қатардағы әр кіріс
    пикселін бір-бірлеп 1-ге қойып, ортаңғы шығыс пиксель өзгеретінін санаймыз."""
    def forward(img):
        z = img[:, :, None]
        for kind, k, s, p, d in layers:
            if kind == "conv":
                z = conv2d(z, np.ones((k, k, 1, 1)), np.zeros(1), stride=s, pad=p, dil=d)
            else:
                z = maxpool2d(z, k, s, p)
        return z[:, :, 0]

    out_shape = forward(np.zeros((n_in, n_in))).shape
    ci, cj = out_shape[0] // 2, out_shape[1] // 2
    row = n_in // 2
    hits = []
    for j in range(n_in):
        img = np.zeros((n_in, n_in))
        img[row, j] = 1
        if forward(img)[ci, cj] != 0:
            hits.append(j)
    return hits[-1] - hits[0] + 1, len(hits)   # (RF өлшемі = аралық, әсер ететін пиксель саны)


layers2 = [("conv", 7, 2, 3, 1), ("pool", 3, 2, 1, 1)]
print("RF (эксперимент, аралық / әсер ететін пиксель):", rf_empirical(layers2, 128))

## Есеп 3

Дилатациясы $d=2$ болатын үш $3\times3$ конволюция, $S=1$.

$$k_{eff}=d(k-1)+1$$

In [ ]:
d, k = 2, 3
k_eff = d * (k - 1) + 1
print(f"K_eff = {k_eff}")

rf, jump = 1, 1
for i in range(3):
    rf += (k_eff - 1) * jump
    print(f"{i+1}-қабаттан кейін RF = {rf}")

layers3 = [("conv", 3, 1, 0, 2)] * 3
print("RF (эксперимент, аралық / әсер ететін пиксель):", rf_empirical(layers3, 64))
print("Дилатацияда RF = 13 аралықты қамтиды, бірақ ядро тесік болғандықтан тек 7 пиксель әсер етеді (gridding)")

## Есеп 4

64 арналы қабаттан кейінгі BatchNorm. Әр арнаға: $\gamma,\beta$ (оқытылады) және running mean/var (статистика).

In [ ]:
bn = BatchNorm(64)
out = bn(y1)     # 1-есептің шығысы, 64 арна
print(f"Оқытылатын (gamma, beta): {bn.n_trainable()}")
print(f"Статистика (running mean, var): {bn.n_stats()}")
print(f"Нормализациядан кейін арна орташасы ~ {out.mean(axis=(0,1)).round(6)[:3]}..., std ~ {out.std(axis=(0,1)).round(3)[:3]}...")

## Қорытынды

| Есеп | Жауап |
|---|---|
| 1 | $64\times64\times64$; 9 472 параметр |
| 2 | $32\times32\times64$; RF = 7, содан кейін 11 |
| 3 | $K_{eff}=5$; RF = 13 |
| 4 | 128 оқытылатын + 128 статистика |